# 00 — Gemini Live API: Getting Started

**Audience**: Python developers new to the Gemini Live API  
**Duration**: ~20 minutes  
**Goal**: Understand the basics of real-time bidirectional streaming with the Gemini Live API

---

## What is the Gemini Live API?

The **Gemini Live API** provides **real-time, bidirectional streaming** between your application and Gemini. Unlike the standard generate-content API (request → wait → response), the Live API works like a persistent phone call:

- You open a **WebSocket session** that stays alive
- You can **send audio, text, or video frames at any time**
- Gemini **responds in real-time** with audio or text chunks as they are generated
- Either side can speak — true **full-duplex** communication

### Why use it?

| Use case | Why Live API fits |
|---|---|
| Voice assistants | Low-latency spoken responses |
| Real-time transcription | Stream audio as it's captured |
| Interactive tutors | Back-and-forth conversation |
| Live video understanding | Send frames and ask questions |
| Call center automation | Bidirectional phone audio |

### How it differs from standard Gemini

```
Standard API:   [send full prompt] ──► [wait] ──► [get full response]

Live API:       [open session]
                    │
                    ├──► send text/audio chunk ──► Gemini receives
                    ├◄── audio/text response ◄──── Gemini responds
                    ├──► send more audio ──────► continues
                    ├◄── more response ◄────────── streams
                    │
                [close session]
```

---

## Prerequisites

- Python 3.9+
- A Gemini API key from [Google AI Studio](https://aistudio.google.com)
- Basic Python async/await knowledge is helpful but not required

## Step 1: Install Dependencies

In [1]:
# Install required packages
# google-genai : the official Google Generative AI Python SDK
# nest_asyncio : allows asyncio.run() to work inside Jupyter (which already has an event loop)
# numpy        : for generating synthetic audio
!pip install google-genai nest_asyncio numpy --quiet

## Step 2: Setup & Authentication

The `nest_asyncio.apply()` call is **required for Jupyter** — Jupyter runs its own asyncio event loop, and without this patch, `asyncio.run()` would raise a `RuntimeError: This event loop is already running`. The patch allows nested event loops.

In [2]:
import nest_asyncio; nest_asyncio.apply()   # Must be first — patches Jupyter's event loop

import asyncio
import os
import numpy as np
import IPython.display as ipd

from google import genai
from google.genai import types

# ── API key ────────────────────────────────────────────────────────────────────
# Get a free key at: https://aistudio.google.com/app/apikey
from dotenv import load_dotenv
load_dotenv()  # loads GEMINI_API_KEY from .env

API_KEY = os.environ.get("GEMINI_API_KEY", "")

# ── Model & client ─────────────────────────────────────────────────────────────
# gemini-3.1-flash-live-preview is the recommended model for Live API sessions.
# It is optimised for low-latency streaming responses.
MODEL  = "gemini-3.1-flash-live-preview"
client = genai.Client(api_key=API_KEY)

print("✓ Setup complete")
print(f"  Model : {MODEL}")
print(f"  Client: {type(client).__name__}")

AttributeError: module 'nest_asyncio' has no attribute 'patch'

## Utility Functions

Two small helpers used throughout this notebook:

- **`make_pcm`** — generates a synthetic sine-wave audio clip as raw PCM16 bytes. This means you can run every demo without needing a microphone or audio files.
- **`play_pcm`** — converts raw PCM16 bytes into an IPython `Audio` widget for in-notebook playback.

In [ ]:
def make_pcm(text_hint: str = "", duration: float = 2.0, rate: int = 16000) -> bytes:
    """
    Generate a sine-wave tone as raw PCM16 bytes.

    Args:
        text_hint : pass "low" to get a 220 Hz tone, otherwise 440 Hz (A4)
        duration  : length of the clip in seconds
        rate      : sample rate in Hz (Gemini input expects 16000)

    Returns:
        bytes : little-endian signed 16-bit PCM samples
    """
    freq = 220 if "low" in text_hint else 440
    t    = np.linspace(0, duration, int(rate * duration), endpoint=False)
    # Amplitude 0.3 keeps it well within 16-bit range and avoids clipping
    samples = (np.sin(2 * np.pi * freq * t) * 0.3 * 32767).astype(np.int16)
    return samples.tobytes()


def play_pcm(raw_bytes: bytes, rate: int = 24000) -> ipd.Audio:
    """
    Wrap raw PCM16 bytes in an IPython Audio widget.

    Args:
        raw_bytes : PCM16 bytes (little-endian int16)
        rate      : sample rate (Gemini outputs at 24000 Hz)

    Returns:
        IPython.display.Audio widget
    """
    arr = np.frombuffer(raw_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    return ipd.Audio(arr, rate=rate, autoplay=False)


print("✓ Helpers defined")
print(f"  make_pcm(duration=1.0) → {len(make_pcm(duration=1.0))} bytes")

---
## Demo 1: Text In → Text Out

The simplest Live API session: send a text message, receive a text reply.

Key points:
- `response_modalities=["TEXT"]` tells Gemini to respond in text (not audio)
- `client.aio.live.connect(...)` opens the WebSocket session as an async context manager
- `session.send_realtime_input(text=...)` sends a text turn
- `session.receive()` is an async generator — each iteration yields one response chunk
- `resp.text` contains the text content of the current chunk (may be `None` for non-text chunks)

In [ ]:
async def demo_text_to_text():
    """
    Open a Live API session, send one text message, collect and print the reply.
    response_modalities=["TEXT"] means Gemini speaks in text, not audio.
    """
    config = types.LiveConnectConfig(
        response_modalities=["TEXT"],   # Request text output
    )

    full_response = []

    # The 'async with' block opens the WebSocket and closes it cleanly on exit
    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print("[Session opened]")

        # ── Send our question ──────────────────────────────────────────────────
        # send_realtime_input is the correct method for Live API sessions.
        # NEVER mix send_client_content with send_realtime_input — causes 1008 error.
        await session.send_realtime_input(
            text="In one sentence, what is the Gemini Live API?"
        )

        # ── Receive response chunks ───────────────────────────────────────────
        # resp.text may be None for non-text events; we filter those out.
        async for resp in session.receive():
            if resp.text:                        # text chunk arrived
                print(resp.text, end="", flush=True)
                full_response.append(resp.text)
            if resp.server_content and resp.server_content.turn_complete:
                print()  # newline after Gemini finishes speaking
                break    # Gemini signalled it's done with this turn
            if resp.go_away:                     # server asks us to disconnect
                print("\n[go_away received — server closing session]")
                break

    print("[Session closed]")
    return "".join(full_response)


print("── Demo 1: Text → Text ──")
result = asyncio.run(demo_text_to_text())
print(f"\nCaptured {len(result)} characters total.")

---
## Demo 2: Text In → Audio Out

Switch the response modality to `"AUDIO"`. Gemini will now synthesise speech and stream it back as raw **PCM16 audio at 24 000 Hz**.

Key points:
- `response_modalities=["AUDIO"]` → Gemini responds in speech
- Audio arrives in `resp.data` as raw bytes (PCM16, 24 kHz, mono)
- We collect all chunks into a list and concatenate them at the end
- `play_pcm()` wraps the bytes in an IPython `Audio` widget

In [ ]:
async def demo_text_to_audio():
    """
    Send a text prompt, receive spoken audio back from Gemini.
    Returns concatenated PCM16 bytes (24 kHz mono).
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],   # Request audio (speech) output
    )

    audio_chunks = []   # collect raw PCM bytes as they arrive

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print("[Session opened — audio mode]")

        # Send a prompt that produces a short, complete sentence
        await session.send_realtime_input(
            text="Say exactly: 'Hello, I am Gemini. The Live API lets you talk to me in real time.'"
        )

        chunk_count = 0
        async for resp in session.receive():
            if resp.data:                        # raw audio bytes arrived
                audio_chunks.append(resp.data)
                chunk_count += 1
                print(f"  ← audio chunk {chunk_count}: {len(resp.data):,} bytes", flush=True)

            if resp.server_content and resp.server_content.turn_complete:
                print("[turn complete]")
                break
            if resp.go_away:
                print("[go_away — disconnecting]")
                break

    print("[Session closed]")
    return b"".join(audio_chunks)


print("── Demo 2: Text → Audio ──")
audio_bytes = asyncio.run(demo_text_to_audio())

total_samples = len(audio_bytes) // 2          # 2 bytes per int16 sample
duration_sec  = total_samples / 24000          # 24 kHz output rate
print(f"\nReceived {len(audio_bytes):,} bytes → {duration_sec:.2f}s of audio")
print("Playing audio:")
play_pcm(audio_bytes, rate=24000)

---
## Demo 3: Synthetic Audio In → Audio Out

Now we send **audio** to Gemini instead of text. This simulates a microphone stream.

Key points:
- Input audio must be **PCM16 at 16 000 Hz** (Gemini's required input format)
- Use `types.Blob(data=pcm_bytes, mime_type="audio/pcm;rate=16000")` to wrap it
- Pass the blob to `session.send_realtime_input(audio=...)`
- Gemini transcribes and processes the audio, then replies in audio

> **Note**: Our synthetic sine-wave tone is not real speech, so Gemini will likely describe what it hears (silence, a tone, or noise) rather than responding to a spoken question. In a real app you would send actual microphone PCM frames.

In [ ]:
async def demo_audio_to_audio():
    """
    Send synthetic PCM audio to Gemini and receive spoken audio back.
    Demonstrates the send_realtime_input(audio=...) pattern.
    """
    config = types.LiveConnectConfig(
        response_modalities=["AUDIO"],
        # System instruction primes Gemini to respond to the audio content
        system_instruction=types.Content(
            parts=[types.Part(text=(
                "You are a helpful assistant. "
                "If the audio you hear is a tone rather than speech, "
                "say: 'I heard a test tone. Send me real speech and I will respond.'"
            ))]
        ),
    )

    # ── Generate test audio ────────────────────────────────────────────────────
    # 2 seconds of 440 Hz sine wave, 16 kHz sample rate (Gemini's required input rate)
    pcm_input = make_pcm(duration=2.0, rate=16000)
    print(f"Sending {len(pcm_input):,} bytes of synthetic audio (2s @ 16kHz PCM16)")

    # Preview what we're sending
    ipd.display(play_pcm(pcm_input, rate=16000))

    audio_chunks = []

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print("[Session opened — audio-in / audio-out]")

        # ── Send audio blob ────────────────────────────────────────────────────
        # types.Blob wraps raw bytes with a MIME type.
        # The mime_type MUST include the sample rate for PCM data.
        audio_blob = types.Blob(
            data=pcm_input,
            mime_type="audio/pcm;rate=16000"
        )
        await session.send_realtime_input(audio=audio_blob)
        print("[Audio sent — waiting for response]")

        async for resp in session.receive():
            if resp.data:
                audio_chunks.append(resp.data)

            if resp.server_content and resp.server_content.turn_complete:
                print("[turn complete]")
                break
            if resp.go_away:
                print("[go_away]")
                break

    print("[Session closed]")
    return b"".join(audio_chunks)


print("── Demo 3: Audio In → Audio Out ──\n")
response_audio = asyncio.run(demo_audio_to_audio())

if response_audio:
    duration_sec = (len(response_audio) // 2) / 24000
    print(f"\nResponse: {len(response_audio):,} bytes ({duration_sec:.2f}s)")
    print("Playing response:")
    play_pcm(response_audio, rate=24000)
else:
    print("No audio returned (check API key or model availability)")

---
## Bonus: Multi-Turn Conversation

The Live API session is **persistent** — you can send multiple messages in a single session without reopening the WebSocket. This enables true conversational back-and-forth.

In [ ]:
async def demo_multi_turn():
    """
    Send three questions in one session, collecting text responses.
    Notice the session stays open between turns — no reconnection cost.
    """
    config = types.LiveConnectConfig(
        response_modalities=["TEXT"],
    )

    questions = [
        "What is 2 + 2? Answer in one word.",
        "What is the capital of France? Answer in one word.",
        "Name one primary colour. Answer in one word.",
    ]

    conversation = []

    async with client.aio.live.connect(model=MODEL, config=config) as session:
        print("[Session opened — multi-turn]\n")

        for i, question in enumerate(questions, 1):
            print(f"Turn {i} → User: {question}")

            await session.send_realtime_input(text=question)

            reply_parts = []
            async for resp in session.receive():
                if resp.text:
                    reply_parts.append(resp.text)
                if resp.server_content and resp.server_content.turn_complete:
                    break
                if resp.go_away:
                    print("[go_away]")
                    break

            reply = "".join(reply_parts).strip()
            print(f"Turn {i} ← Gemini: {reply}\n")
            conversation.append({"user": question, "gemini": reply})

    print("[Session closed]")
    return conversation


print("── Bonus: Multi-Turn Conversation ──\n")
conv = asyncio.run(demo_multi_turn())
print(f"\nCompleted {len(conv)} turns in a single session.")

---
## Session Lifecycle & Limits

### How a session ends

There are three ways a Live API session terminates:

1. **Normal close** — your code exits the `async with` block. The SDK sends a clean WebSocket close frame.
2. **`resp.go_away`** — the server sends a `GoAway` message signalling it will close the session soon (e.g., session time limit reached, server maintenance). You should stop sending, drain remaining responses, and reconnect.
3. **Error** — a WebSocket or protocol error (e.g., mixing `send_client_content` with `send_realtime_input`).

### Session limits (as of mid-2025)

| Limit | Value |
|---|---|
| Max session duration | ~15 minutes |
| Max concurrent sessions | Depends on quota tier |
| Input audio sample rate | 16 000 Hz (PCM16) |
| Output audio sample rate | 24 000 Hz (PCM16) |

### Reconnect pattern

```python
# Pseudocode for resilient reconnect
while True:
    async with client.aio.live.connect(model=MODEL, config=config) as session:
        async for resp in session.receive():
            if resp.go_away:
                break   # exit inner loop, outer while opens a new session
            # ... handle resp ...
```

### Critical rule — don't mix send methods

```python
# ✅ CORRECT — use send_realtime_input for Live API sessions
await session.send_realtime_input(text="Hello")
await session.send_realtime_input(audio=blob)

# ❌ WRONG — send_client_content causes WebSocket error 1008
await session.send_client_content(turns=[...])   # do NOT use this
```

---
## Quick Reference Card

In [ ]:
# ── Quick-reference: all the patterns in one cell ──────────────────────────────

# 1. Open a session
#    async with client.aio.live.connect(model=MODEL, config=config) as session:

# 2. Send text
#    await session.send_realtime_input(text="Hello, Gemini")

# 3. Send audio
#    blob = types.Blob(data=pcm16_bytes, mime_type="audio/pcm;rate=16000")
#    await session.send_realtime_input(audio=blob)

# 4. Receive responses
#    async for resp in session.receive():
#        resp.text          → text chunk (or None)
#        resp.data          → raw PCM audio bytes (or None)
#        resp.server_content.turn_complete → True when Gemini finishes a turn
#        resp.tool_call     → tool call request (for function calling, see later notebooks)
#        resp.go_away       → server closing, reconnect needed

# 5. Respond to tool calls
#    await session.send_tool_response(function_responses=[...])

print("Reference card loaded — no output expected from this cell.")
print("Read the comments above.")

---
## Key Takeaways

1. **The Live API is bidirectional streaming** — one open WebSocket, send and receive simultaneously.

2. **Always use `nest_asyncio.apply()`** in Jupyter before any `asyncio.run()` calls.

3. **`send_realtime_input`** is the correct send method. Never use `send_client_content` in the same session.

4. **Text or audio input, text or audio output** — controlled by `response_modalities` in the config.

5. **Audio format**: input must be PCM16 @ 16 kHz; output arrives as PCM16 @ 24 kHz.

6. **One session, many turns** — keep the session open for a full conversation rather than reconnecting per turn.

7. **Handle `go_away`** — always check `resp.go_away` and reconnect if needed.

---
**Next notebook →** `01_audio_streaming.ipynb` — WAV files, chunked streaming, and audio format conversion